In [2]:
from pyspark.sql import functions as F

bronze_path = "Files/bronze/appointment/appointment_demo.ndjson"

df_raw = spark.read.json(bronze_path)

display(df_raw)
df_raw.printSchema()

StatementMeta(, 2138e460-4d4b-4f93-a915-d2e47504d98b, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a10d5f0c-3cfe-4f5d-9c6e-bf6c79d20964)

root
 |-- appointmentType: struct (nullable = true)
 |    |-- coding: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- code: string (nullable = true)
 |    |    |    |-- display: string (nullable = true)
 |    |    |    |-- system: string (nullable = true)
 |-- description: string (nullable = true)
 |-- end: string (nullable = true)
 |-- id: string (nullable = true)
 |-- meta: struct (nullable = true)
 |    |-- lastUpdated: string (nullable = true)
 |-- participant: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- actor: struct (nullable = true)
 |    |    |    |-- display: string (nullable = true)
 |    |    |-- status: string (nullable = true)
 |-- reasonCode: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- text: string (nullable = true)
 |-- resourceType: string (nullable = true)
 |-- serviceCategory: array (nullable = true)
 |    |-- element: struct (contains

In [3]:
df_raw.write.mode("overwrite").saveAsTable("bronze_appointments")

StatementMeta(, 2138e460-4d4b-4f93-a915-d2e47504d98b, 5, Finished, Available, Finished, False)

In [6]:
df_silver = df_raw.select(
    F.col("id").alias("appointment_id"),
    F.col("resourceType").alias("resource_type"),
    F.col("status"),
    F.col("description"),
    F.to_timestamp("start").alias("start_time"),
    F.to_timestamp("end").alias("end_time"),
    F.to_timestamp("meta.lastUpdated").alias("last_updated"),
    F.col("participant")[0]["actor"]["display"].alias("provider_name"),
    F.col("participant")[1]["actor"]["display"].alias("patient_name"),
    F.col("participant")[0]["status"].alias("provider_status"),
    F.col("participant")[1]["status"].alias("patient_status"),
    F.col("serviceCategory")[0]["coding"][0]["code"].alias("service_category_code"),
    F.col("serviceCategory")[0]["coding"][0]["display"].alias("service_category"),
    F.col("appointmentType")["coding"][0]["code"].alias("appointment_type_code"),
    F.col("appointmentType")["coding"][0]["display"].alias("appointment_type"),
    F.col("reasonCode")[0]["text"].alias("reason_text")
)

display(df_silver)

StatementMeta(, 2138e460-4d4b-4f93-a915-d2e47504d98b, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b5552a06-30eb-42e2-933c-f9badadc6d01)

In [7]:
df_silver.write.mode("overwrite").saveAsTable("silver_appointments")

StatementMeta(, 2138e460-4d4b-4f93-a915-d2e47504d98b, 9, Finished, Available, Finished, False)

In [8]:
df_gold = (
    df_silver
    .groupBy("service_category", "status")
    .count()
    .orderBy("service_category", "status")
)

display(df_gold)

StatementMeta(, 2138e460-4d4b-4f93-a915-d2e47504d98b, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0b758a93-2caa-4266-b79e-744b31b95430)

In [9]:
df_gold.write.mode("overwrite").saveAsTable("gold_appointment_summary")

StatementMeta(, 2138e460-4d4b-4f93-a915-d2e47504d98b, 11, Finished, Available, Finished, False)